# 01 — Exploratory Data Analysis

This notebook connects to the raw Divvy trip parquet file via DuckDB and performs
an initial schema inspection. It examines null counts, data types, and categorical
distributions to build a working understanding of the dataset before any feature
engineering is applied.

## Assumptions & Dependencies

- The raw parquet file (`divvy.parquet`) must exist at the path provided when prompted.
- No prior notebook needs to have been run, this is the first stage in the pipeline.
- Required packages: `duckdb`, `pandas`, `matplotlib` (see `requirements.txt`).

## Setup — load data path and connect DuckDB

We prompt for the file path at runtime so the notebook works on any machine
without hardcoded paths. The `connect_duckdb` helper registers the parquet file
as a DuckDB view named `trips`.

In [2]:
import sys
sys.path.insert(0, '..') # for future imports to access files outside of this directory

from src.utils import get_data_path, connect_duckdb

file_path = get_data_path()
con = connect_duckdb(file_path)

## Schema Inspection

Inspect column names, data types, and the total row count to confirm the parquet
file loaded correctly and matches expectations.

In [6]:
# Schema and row count
print(con.execute("DESCRIBE trips").df())
print("\nRow count:", con.execute("SELECT COUNT(*) FROM trips").fetchone()[0])

          column_name column_type null   key default extra
0             trip_id      BIGINT  YES  None    None  None
1                year      BIGINT  YES  None    None  None
2               month      BIGINT  YES  None    None  None
3                week      BIGINT  YES  None    None  None
4                 day      BIGINT  YES  None    None  None
5                hour      BIGINT  YES  None    None  None
6            usertype     VARCHAR  YES  None    None  None
7              gender     VARCHAR  YES  None    None  None
8           starttime   TIMESTAMP  YES  None    None  None
9            stoptime   TIMESTAMP  YES  None    None  None
10       tripduration      DOUBLE  YES  None    None  None
11        temperature      DOUBLE  YES  None    None  None
12             events     VARCHAR  YES  None    None  None
13    from_station_id      BIGINT  YES  None    None  None
14  from_station_name     VARCHAR  YES  None    None  None
15     latitude_start      DOUBLE  YES  None    None  No

## Null Counts

Count nulls per column to identify which fields will need imputation or special
handling in the feature engineering stage.

In [4]:
# Null counts per column
null_query = """
SELECT
    COUNT(*) - COUNT(trip_id)          AS trip_id_nulls,
    COUNT(*) - COUNT(starttime)        AS starttime_nulls,
    COUNT(*) - COUNT(stoptime)         AS stoptime_nulls,
    COUNT(*) - COUNT(from_station_id)  AS from_station_id_nulls,
    COUNT(*) - COUNT(to_station_id)    AS to_station_id_nulls,
    COUNT(*) - COUNT(temperature)      AS temperature_nulls,
    COUNT(*) - COUNT(events)           AS events_nulls
FROM trips
"""
con.execute(null_query).df()

,trip_id_nulls,starttime_nulls,stoptime_nulls,from_station_id_nulls,to_station_id_nulls,temperature_nulls,events_nulls
0,0,0,0,0,0,0,0


## Categorical Distributions

Examine the distribution of key categorical fields — station IDs, weather events,
and user types — to understand cardinality and class balance.

In [5]:
# Departure only vs arrival only vs both
station_overlap_query = """
WITH departures AS (
    SELECT DISTINCT from_station_id AS station_id FROM trips
),
arrivals AS (
    SELECT DISTINCT to_station_id AS station_id FROM trips
)
SELECT
    COUNT(CASE WHEN a.station_id IS NULL THEN 1 END) AS departure_only,
    COUNT(CASE WHEN d.station_id IS NULL THEN 1 END) AS arrival_only,
    COUNT(CASE WHEN d.station_id IS NOT NULL AND a.station_id IS NOT NULL THEN 1 END) AS both
FROM departures d
FULL OUTER JOIN arrivals a ON d.station_id = a.station_id
"""
con.execute(station_overlap_query).df()

,departure_only,arrival_only,both
0,0,0,586


In [7]:
print(con.execute("SELECT COUNT(DISTINCT from_station_id) AS station_count FROM trips").df())
print(con.execute("SELECT MIN(starttime)::DATE as min_date, MAX(starttime)::DATE as max_date FROM trips").df())

   station_count
0            586
    min_date   max_date
0 2014-01-01 2017-12-31


In [9]:
# Weather events distribution
con.execute("SELECT events, COUNT(*) as trip_count FROM trips GROUP BY events ORDER BY trip_count DESC").df()

,events,trip_count
0,cloudy,8398501
1,clear,511819
2,rain or snow,432077
3,not clear,88159
4,tstorms,64143
5,unknown,536


## Summary

**Outputs produced by this notebook:** None — this is a read-only exploration.

**What the next notebook expects:** `02_feature_engineering.ipynb` assumes the
same parquet file is accessible at the path returned by `get_data_path()` and
that the `trips` view schema has been confirmed here.